# 02 - Análise Exploratória

Este notebook apresenta estatística exploratória e visualizações simples para entender a base de clientes, transações e categorias da Aurora.

A proposta é mostrar o processo analítico de forma transparente para a banca: primeiro descrevemos os dados, depois conectamos os números a interpretações de negócio.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DADOS_RAW = ROOT / "dados" / "raw"
DADOS_PROCESSED = ROOT / "dados" / "processed"
DADOS_OUTPUTS = ROOT / "dados" / "outputs"
print(f"Raiz do projeto: {ROOT}")

In [ ]:
clientes = pd.read_csv(DADOS_PROCESSED / "clientes_limpo.csv", parse_dates=["data_cadastro"])
transacoes = pd.read_csv(DADOS_PROCESSED / "transacoes_limpo.csv", parse_dates=["data"])
categorias = pd.read_csv(DADOS_PROCESSED / "categorias_limpo.csv")

transacoes_cat = transacoes.merge(categorias, on="categoria_id", how="left")
print(clientes.shape, transacoes.shape, categorias.shape)

## 1. Estatísticas descritivas dos clientes

Começamos com média, mediana, desvio padrão e percentis das variáveis numéricas. Isso ajuda a entender escala, dispersão e possíveis extremos.

In [ ]:
cols_num = ["idade", "renda_mensal", "saldo_atual", "score_credito", "tempo_relacionamento", "produtos_ativos"]
descritivas = clientes[cols_num].describe(percentiles=[.05, .25, .5, .75, .95]).T
descritivas["mediana"] = clientes[cols_num].median()
descritivas["desvio_padrao"] = clientes[cols_num].std()
display(descritivas)

**Interpretação de negócio:** idade, renda, saldo e score de crédito ajudam a segmentar perfis financeiros. A banca deve observar se existem faixas de renda ou score mais associadas a churn e risco.

## 2. Taxa de churn observada

Quando o dataset público está disponível, `churn_flag` vem de `Exited`. No fallback, a variável é sintética, mas mantém o mesmo papel analítico.

In [ ]:
taxa_churn = clientes["churn_flag"].mean()
print(f"Taxa de churn observada: {taxa_churn:.2%}")
display(clientes["churn_flag"].value_counts(normalize=True).rename("proporcao").reset_index())

**Interpretação de negócio:** a taxa de churn define o tamanho do problema. Se muitos clientes saem, o modelo ajuda a priorizar retenção; se poucos saem, precision e recall precisam ser avaliados com cuidado.

## 3. Distribuições de idade, renda, saldo e score

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["idade", "renda_mensal", "saldo_atual", "score_credito"]):
    ax.hist(clientes[col].dropna(), bins=30, color="#20c7df", edgecolor="white")
    ax.set_title(f"Distribuição de {col}")
    ax.set_ylabel("Clientes")
plt.tight_layout()
plt.show()

**Interpretação de negócio:** histogramas mostram concentração e extremos. Saldos muito baixos ou muito altos parados podem indicar perfis distintos de relacionamento e risco.

## 4. Análise por estado/localização

In [ ]:
por_estado = clientes.groupby("estado").agg(
    clientes=("cliente_id", "nunique"),
    churn_rate=("churn_flag", "mean"),
    renda_media=("renda_mensal", "mean"),
    saldo_medio=("saldo_atual", "mean"),
).sort_values("clientes", ascending=False)
display(por_estado)

por_estado["clientes"].plot(kind="bar", figsize=(8, 4), color="#8558f2", title="Clientes por estado")
plt.ylabel("Clientes")
plt.tight_layout()
plt.show()

**Interpretação de negócio:** a análise regional ajuda a decidir onde campanhas de retenção e educação financeira podem ser priorizadas.

## 5. Análise por perfil de risco

In [ ]:
perfil = clientes.groupby("perfil_risco").agg(
    clientes=("cliente_id", "nunique"),
    churn_rate=("churn_flag", "mean"),
    score_medio=("score_credito", "mean"),
    renda_media=("renda_mensal", "mean"),
).sort_values("churn_rate", ascending=False)
display(perfil)

perfil["churn_rate"].plot(kind="bar", figsize=(8, 4), color="#ff6b7a", title="Taxa de churn por perfil de risco")
plt.ylabel("Taxa de churn")
plt.tight_layout()
plt.show()

**Interpretação de negócio:** o perfil de risco resume sinais de score, saldo, renda e comportamento. Ele é útil para comunicação executiva e segmentação no Power BI.

## 6. Top categorias de consumo

In [ ]:
top_categorias = transacoes_cat.groupby("nome_categoria")["valor"].sum().sort_values(ascending=False).head(10)
display(top_categorias.reset_index(name="volume_financeiro"))

top_categorias.sort_values().plot(kind="barh", figsize=(9, 5), color="#f8a400", title="Top categorias por volume financeiro")
plt.xlabel("Volume financeiro")
plt.tight_layout()
plt.show()

**Interpretação de negócio:** categorias de maior volume indicam onde esté a maior movimentação financeira e podem orientar parcerias, benefícios e ações de retenção.

## 7. Evolução mensal do volume financeiro

In [ ]:
transacoes_cat["ano_mes"] = transacoes_cat["data"].dt.to_period("M").astype(str)
evolucao = transacoes_cat.groupby("ano_mes")["valor"].sum()
display(evolucao.reset_index(name="volume_financeiro").tail(12))

evolucao.plot(kind="line", marker="o", figsize=(10, 4), color="#20c7df", title="Evolução mensal do volume financeiro")
plt.ylabel("Volume financeiro")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretação de negócio:** a evolução mensal ajuda a contar a história da base: estabilidade, sazonalidade, queda de engajamento ou períodos de maior movimentação.